Прогнозирование вероятности выпадения осадков на следующий день (RainTomorrow) в Австралии.
(Определяет, какие алгоритмы использовать (классификация vs регрессия（поак нет）))
(Задаёт метрику оптимизации (auc для binary, RMSE для regression(пока нет)))

In [1]:
# ==========================================
# ШАГ 0: Импорт библиотек и настройки
# ==========================================

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

print("✅ Шаг 0: Библиотеки успешно импортированы.")

✅ Шаг 0: Библиотеки успешно импортированы.


In [2]:
# ==========================================
# ШАГ 1: Загрузка, Диагностика и Очистка
# ==========================================

from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split

filename = "weatherAUS.csv"
target = 'RainTomorrow'

if os.path.exists(filename):
    print("🔄 1. Загрузка данных (Loading Raw Data)...")
    raw_df = pd.read_csv(filename)

    # A. Генерация отчета
    print("\n📊 2. Генерация глубокого отчета EDA (minimal=False)...")
    profile = ProfileReport(raw_df, title="EDA - Full Raw Data", minimal=False)
    profile.to_file("report_raw_full.html")
    print("✅ Отчет сохранен! (report_raw_full.html)")

    # B. Диагностика
    print("\n🧐 3. Диагностика данных:")
    print(f"   -> [Размер] Всего строк: {raw_df.shape[0]}, Столбцов: {raw_df.shape[1]}")

    duplicates_count = raw_df.duplicated().sum()
    print(f"   -> [Дубликаты] Найдено: {duplicates_count}")

    missing_target = raw_df[target].isna().sum()
    print(f"   -> [Таргет] Пропусков в '{target}': {missing_target}")

    print("   -> [Выбросы] Проверка (Метод IQR):")
    numeric_cols = raw_df.select_dtypes(include=[np.number]).columns
    outlier_info = []
    for col in numeric_cols:
        Q1 = raw_df[col].quantile(0.25)
        Q3 = raw_df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((raw_df[col] < (Q1 - 1.5 * IQR)) | (raw_df[col] > (Q3 + 1.5 * IQR))).sum()
        if outliers > 0:
            outlier_info.append((col, outliers))
    outlier_info.sort(key=lambda x: x[1], reverse=True)
    for col, count in outlier_info[:3]:
        print(f"      - {col}: {count} потенциальных выбросов")

    # C. Очистка
    print("\n🛠️ 4. Очистка:")
    if duplicates_count > 0:
        print(f"   -> 🗑️ Удаление {duplicates_count} дубликатов...")
        df = raw_df.drop_duplicates()
    else:
        df = raw_df.copy()

    print(f"   -> 🗑️ Удаление {missing_target} строк без таргета...")
    df = df.dropna(subset=[target])
    print(f"   -> ✅ Итоговый размер: {df.shape}")

    # D. Разделение на Train/Test
    print("\n✂️ 5. Разделение на Train/Test (Stratified)...")
    train_raw, test_raw = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df[target]
    )
    print(f"   -> Train: {train_raw.shape}, Test: {test_raw.shape}")

    # =========================================================
    # ★★★ 5.5 КЛЮЧЕВОЕ ДОБАВЛЕНИЕ: Сохранение RAW для LAMA ★★★
    # =========================================================
    # LAMA сама обрабатывает: пропуски, категории, даты
    # Поэтому сохраняем данные ДО clean_data()!
    print("\n📦 5.5 Сохранение RAW данных для LightAutoML (ДО заполнения пропусков)...")
    train_raw.to_csv("train_raw_for_lama.csv", index=False)
    test_raw.to_csv("test_raw_for_lama.csv", index=False)
    print("   -> ✅ Сохранены: train_raw_for_lama.csv, test_raw_for_lama.csv")
    print("   -> (С пропусками ✓, с колонкой Date ✓, с исходными категориями ✓)")

    # E. Заполнение пропусков (ТОЛЬКО для традиционных моделей!)
    print("\n🧹 6. Заполнение пропусков (для традиционных моделей)...")

    def clean_data(data):
        d = data.copy()
        if 'Date' in d.columns:
            d['Date'] = pd.to_datetime(d['Date'])
            d['Year'] = d['Date'].dt.year
            d['Month'] = d['Date'].dt.month
            d['Day'] = d['Date'].dt.day
            d = d.drop(columns=['Date'])

        num_cols = d.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            d[col] = d[col].fillna(d[col].median())

        cat_cols = d.select_dtypes(include=['object']).columns
        for col in cat_cols:
            if d[col].isna().sum() > 0:
                d[col] = d[col].fillna(d[col].mode()[0])
        return d

    train_cleaned = clean_data(train_raw)
    test_cleaned = clean_data(test_raw)

    # F. Сохранение
    print("\n💾 7. Сохранение результатов...")
    train_cleaned.to_csv("train_cleaned.csv", index=False)
    test_cleaned.to_csv("test_cleaned.csv", index=False)
    print("✅ Готово!")
    print("   -> train_cleaned.csv / test_cleaned.csv — для традиционных моделей")
    print("   -> train_raw_for_lama.csv / test_raw_for_lama.csv — для LightAutoML")

else:
    print(f"❌ Ошибка: Файл {filename} не найден.")

🔄 1. Загрузка данных (Loading Raw Data)...

📊 2. Генерация глубокого отчета EDA (minimal=False)...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 47.34it/s]


✅ Отчет сохранен! (report_raw_full.html)

🧐 3. Диагностика данных:
   -> [Размер] Всего строк: 145460, Столбцов: 23
   -> [Дубликаты] Найдено: 0
   -> [Таргет] Пропусков в 'RainTomorrow': 3267
   -> [Выбросы] Проверка (Метод IQR):
      - Rainfall: 25578 потенциальных выбросов
      - WindGustSpeed: 3092 потенциальных выбросов
      - WindSpeed3pm: 2523 потенциальных выбросов

🛠️ 4. Очистка:
   -> 🗑️ Удаление 3267 строк без таргета...
   -> ✅ Итоговый размер: (142193, 23)

✂️ 5. Разделение на Train/Test (Stratified)...
   -> Train: (113754, 23), Test: (28439, 23)

📦 5.5 Сохранение RAW данных для LightAutoML (ДО заполнения пропусков)...
   -> ✅ Сохранены: train_raw_for_lama.csv, test_raw_for_lama.csv
   -> (С пропусками ✓, с колонкой Date ✓, с исходными категориями ✓)

🧹 6. Заполнение пропусков (для традиционных моделей)...

💾 7. Сохранение результатов...
✅ Готово!
   -> train_cleaned.csv / test_cleaned.csv — для традиционных моделей
   -> train_raw_for_lama.csv / test_raw_for_lama.csv 

In [3]:
# ==========================================
# ШАГ 2: Глубокая проверка и Генерация отчета
# ==========================================

from ydata_profiling import ProfileReport

print("🔄 Загрузка очищенных данных...")
train_df = pd.read_csv("train_cleaned.csv")
test_df = pd.read_csv("test_cleaned.csv")

def verify_dataset(df, name):
    print(f"\n🧐 --- Проверка: [{name}] ---")
    print(f"   1. Размер: {df.shape}")
    nans = df.isna().sum().sum()
    print(f"   2. Пропуски: {'✅ Чисто' if nans == 0 else f'⚠️ {nans}'}")
    dups = df.duplicated().sum()
    print(f"   3. Дубликаты: {'✅ Нет' if dups == 0 else f'⚠️ {dups}'}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    total_outliers = 0
    for col in numeric_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        total_outliers += ((df[col] < (Q1 - 1.5*IQR)) | (df[col] > (Q3 + 1.5*IQR))).sum()
    print(f"   4. Выбросы: {total_outliers} (норма для погодных данных)")

verify_dataset(train_df, "Train")
verify_dataset(test_df, "Test")

target = 'RainTomorrow'
if target in train_df.columns:
    print(f"\n⚖️ Баланс классов в Train ('{target}'):")
    print(train_df[target].value_counts(normalize=True))

print("\n📊 Генерация отчетов EDA...")
profile_train = ProfileReport(train_df, title="EDA - Cleaned Train", minimal=False, explorative=True)
profile_train.to_file("report_train.html")
profile_test = ProfileReport(test_df, title="EDA - Cleaned Test", minimal=False, explorative=True)
profile_test.to_file("report_test.html")
print("\n✅ Отчеты созданы.")

🔄 Загрузка очищенных данных...

🧐 --- Проверка: [Train] ---
   1. Размер: (113754, 25)
   2. Пропуски: ✅ Чисто
   3. Дубликаты: ✅ Нет
   4. Выбросы: 115462 (норма для погодных данных)

🧐 --- Проверка: [Test] ---
   1. Размер: (28439, 25)
   2. Пропуски: ✅ Чисто
   3. Дубликаты: ✅ Нет
   4. Выбросы: 29125 (норма для погодных данных)

⚖️ Баланс классов в Train ('RainTomorrow'):
RainTomorrow
No     0.775814
Yes    0.224186
Name: proportion, dtype: float64

📊 Генерация отчетов EDA...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 29.28it/s]


✅ Отчеты созданы.


In [4]:
# ==========================================
# ШАГ 3: Подготовка данных и Кодирование
# ==========================================

import category_encoders as ce
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("🔄 [Шаг 3] Чтение данных...")
train_df = pd.read_csv("train_cleaned.csv")
test_df = pd.read_csv("test_cleaned.csv")

target_col = 'RainTomorrow'

if train_df[target_col].dtype == 'object':
    print("   -> Кодирование таргета...")
    le = LabelEncoder()
    train_df[target_col] = le.fit_transform(train_df[target_col])
    test_df[target_col] = le.transform(test_df[target_col])

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"📝 Категориальные признаки: {categorical_cols}")

print("\n⚙️ Применение кодировщиков...")

woe_encoder = ce.WOEEncoder(cols=categorical_cols)
X_train_woe = woe_encoder.fit_transform(X_train, y_train)
X_test_woe = woe_encoder.transform(X_test)

te_encoder = ce.TargetEncoder(cols=categorical_cols)
X_train_te = te_encoder.fit_transform(X_train, y_train)
X_test_te = te_encoder.transform(X_test)

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_woe)
X_test_lr = scaler.transform(X_test_woe)

print("✅ Шаг 3 выполнен.")

🔄 [Шаг 3] Чтение данных...
   -> Кодирование таргета...
📝 Категориальные признаки: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']

⚙️ Применение кодировщиков...
✅ Шаг 3 выполнен.


In [5]:
# ==========================================
# ШАГ 4: Обучение традиционных моделей
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

print("🔥 [Шаг 4] Обучение традиционных моделей...")

predictions = {}
trained_models = {}

# Модель 1: Logistic Regression
print("\n1️⃣ Logistic Regression (WoE данные)...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_lr, y_train)
predictions['Logistic Regression'] = lr_model.predict_proba(X_test_lr)[:, 1]
trained_models['Logistic Regression'] = {
    'model': lr_model, 'X_test': X_test_lr, 'encoding': 'WoE'
}

# Модель 2: Random Forest
print("2️⃣ Random Forest (TargetEnc данные)...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
rf_model.fit(X_train_te, y_train)
predictions['Random Forest'] = rf_model.predict_proba(X_test_te)[:, 1]
trained_models['Random Forest'] = {
    'model': rf_model, 'X_test': X_test_te, 'encoding': 'TargetEnc'
}

# Модель 3: XGBoost
print("3️⃣ XGBoost (TargetEnc данные)...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_te, y_train)
predictions['XGBoost'] = xgb_model.predict_proba(X_test_te)[:, 1]
trained_models['XGBoost'] = {
    'model': xgb_model, 'X_test': X_test_te, 'encoding': 'TargetEnc'
}

print("\n📊 Результаты традиционных моделей:")
for name, preds in predictions.items():
    print(f"   -> {name}: AUC = {roc_auc_score(y_test, preds):.4f}")

print("\n✅ Шаг 4 выполнен: 3 традиционные модели обучены.")

🔥 [Шаг 4] Обучение традиционных моделей...

1️⃣ Logistic Regression (WoE данные)...
2️⃣ Random Forest (TargetEnc данные)...
3️⃣ XGBoost (TargetEnc данные)...

📊 Результаты традиционных моделей:
   -> Logistic Regression: AUC = 0.8673
   -> Random Forest: AUC = 0.8744
   -> XGBoost: AUC = 0.8979

✅ Шаг 4 выполнен: 3 традиционные модели обучены.


In [6]:
# ==========================================
# ШАГ 4.5: LightAutoML — 3 эксперимента
# ★★★ ИСПОЛЬЗУЕМ RAW ДАННЫЕ! ★★★
# ==========================================

from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

print("\n" + "=" * 65)
print("🤖 [Шаг 4.5] LightAutoML — 3 эксперимента")
print("   ★ ИСПОЛЬЗУЕМ НЕОБРАБОТАННЫЕ ДАННЫЕ (raw)!")
print("   ★ LAMA сама обработает пропуски, категории и даты!")
print("=" * 65)

# ★★★ КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: Читаем RAW данные, НЕ cleaned! ★★★
lama_train = pd.read_csv("train_raw_for_lama.csv")
lama_test = pd.read_csv("test_raw_for_lama.csv")

# Кодируем таргет Yes/No → 1/0 (LAMA нужен числовой target для binary)
if lama_train[target_col].dtype == 'object':
    lama_train[target_col] = lama_train[target_col].map({'No': 0, 'Yes': 1})
    lama_test[target_col] = lama_test[target_col].map({'No': 0, 'Yes': 1})

# Преобразуем Date в datetime (LAMA умеет сама извлекать признаки из дат!)
if 'Date' in lama_train.columns:
    lama_train['Date'] = pd.to_datetime(lama_train['Date'])
    lama_test['Date'] = pd.to_datetime(lama_test['Date'])

# y_test для оценки LAMA (те же самые строки, тот же split)
y_test_lama = lama_test[target_col]

print(f"\n📋 Данные для LAMA:")
print(f"   -> Train: {lama_train.shape}")
print(f"   -> Test:  {lama_test.shape}")
print(f"   -> Пропуски в train: {lama_train.isna().sum().sum()}")
print(f"   -> Пропуски в test:  {lama_test.isna().sum().sum()}")
print(f"   -> Колонка 'Date': {'✅ Есть' if 'Date' in lama_train.columns else '❌ Нет'}")
print(f"   -> (LAMA обработает ВСЁ автоматически!)")

task = Task('binary')
roles = {'target': target_col}
lama_results = {}

# ========================
# Эксперимент 1: TabularAutoML, timeout=300
# ========================
print("\n🧪 Эксперимент 1: TabularAutoML (timeout=300с, базовый пресет)...")
automl_1 = TabularAutoML(
    task=task,
    timeout=300,
    cpu_limit=-1,
    reader_params={'n_jobs': 1, 'random_state': 42}
)
oof_pred_1 = automl_1.fit_predict(lama_train, roles=roles, verbose=0)
test_pred_1 = automl_1.predict(lama_test)
lama_preds_1 = test_pred_1.data[:, 0]
auc_1 = roc_auc_score(y_test_lama, lama_preds_1)
lama_results['LAMA_Exp1_Base_300s'] = {
    'preds': lama_preds_1, 'auc': auc_1, 'automl': automl_1
}
print(f"   -> AUC = {auc_1:.4f}")

# ========================
# Эксперимент 2: TabularAutoML, timeout=600
# ========================
print("\n🧪 Эксперимент 2: TabularAutoML (timeout=600с, больше времени)...")
automl_2 = TabularAutoML(
    task=task,
    timeout=600,
    cpu_limit=-1,
    reader_params={'n_jobs': 1, 'random_state': 42}
)
oof_pred_2 = automl_2.fit_predict(lama_train, roles=roles, verbose=0)
test_pred_2 = automl_2.predict(lama_test)
lama_preds_2 = test_pred_2.data[:, 0]
auc_2 = roc_auc_score(y_test_lama, lama_preds_2)
lama_results['LAMA_Exp2_Extended_600s'] = {
    'preds': lama_preds_2, 'auc': auc_2, 'automl': automl_2
}
print(f"   -> AUC = {auc_2:.4f}")

# ========================
# Эксперимент 3: TabularUtilizedAutoML, timeout=600
# ========================
print("\n🧪 Эксперимент 3: TabularUtilizedAutoML (timeout=600с, продвинутый)...")
automl_3 = TabularUtilizedAutoML(
    task=task,
    timeout=600,
    cpu_limit=-1,
    reader_params={'n_jobs': 1, 'random_state': 42}
)
oof_pred_3 = automl_3.fit_predict(lama_train, roles=roles, verbose=0)
test_pred_3 = automl_3.predict(lama_test)
lama_preds_3 = test_pred_3.data[:, 0]
auc_3 = roc_auc_score(y_test_lama, lama_preds_3)
lama_results['LAMA_Exp3_Utilized_600s'] = {
    'preds': lama_preds_3, 'auc': auc_3, 'automl': automl_3
}
print(f"   -> AUC = {auc_3:.4f}")

# Сравнение экспериментов
print("\n" + "=" * 55)
print("📊 Сравнение 3-х экспериментов LightAutoML:")
print("=" * 55)
print(f"{'Эксперимент':<35} {'ROC-AUC':>10}")
print("-" * 47)
for name, res in lama_results.items():
    print(f"   {name:<33} {res['auc']:>10.4f}")

best_lama_name = max(lama_results, key=lambda k: lama_results[k]['auc'])
best_lama = lama_results[best_lama_name]
print(f"\n🏆 Лучший: {best_lama_name} (AUC={best_lama['auc']:.4f})")

# Добавляем лучший LAMA в общий словарь
predictions['LightAutoML (best)'] = best_lama['preds']

# Анализ лучшей модели LAMA
print("\n🔍 Анализ лучшей модели LightAutoML:")

try:
    fast_fi = best_lama['automl'].get_feature_scores('fast')
    print("\n📈 Top-10 признаков по важности:")
    top_fi = fast_fi.set_index('Feature').sort_values('Importance', ascending=False).head(10)
    print(top_fi.to_string())

    fig_fi, ax_fi = plt.subplots(figsize=(10, 6))
    top_fi['Importance'].plot.barh(ax=ax_fi, color='steelblue')
    ax_fi.set_xlabel('Importance')
    ax_fi.set_title(f'Top-10 Feature Importance — {best_lama_name}')
    ax_fi.invert_yaxis()
    plt.tight_layout()
    fig_fi.savefig('lama_feature_importance.png', dpi=300, bbox_inches='tight')
    print("✅ График сохранен: lama_feature_importance.png")
    plt.show()
except Exception as e:
    print(f"⚠️ Feature Importance: {e}")

# Структура модели
print("\n📋 Внутренняя структура лучшей LAMA модели:")
try:
    for i, level in enumerate(best_lama['automl'].levels):
        for j, pipe in enumerate(level):
            algo_name = pipe.ml_algos[0].__class__.__name__
            print(f"   Уровень {i+1}, Пайплайн {j+1}: {algo_name}")
except Exception as e:
    print(f"   (Не удалось: {e})")

print("\n✅ Шаг 4.5 выполнен.")

'nlp' extra dependency package 'fasttext-numpy2' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.

🤖 [Шаг 4.5] LightAutoML — 3 эксперимента
   ★ ИСПОЛЬЗУЕМ НЕОБРАБОТАННЫЕ ДАННЫЕ (raw)!
   ★ LAMA сама обработает пропуски, категории и даты!

📋 Данные для LAMA:
   -> Train: (113754, 23)
   -> Test:  (28439, 23)
   -> Пропуски в train: 253678
   -> Пропуски в test:  62881
   -> Колонка 'Date': ✅ Есть
   -> (LAMA обработает ВСЁ автоматически!)

🧪 Эксперимент 1: TabularAutoML (timeout=300с, базовый пресет)...
   -> AUC = 0.8994

🧪 Эксперимент 2: TabularAutoML (timeout=600с, больше времени)...
   -> AUC = 0.8994

🧪 Эксперимент 3: TabularUtilizedAutoML (timeout=600с, продвинутый)...
   -> AUC = 0.905

In [7]:
# ==========================================
# ШАГ 5: Сводная таблица + Визуализация ROC
# ==========================================

from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score

print("\n📈 [Шаг 5] Сводная таблица + ROC-кривые...")

# 5A. Сводная таблица
print("\n" + "=" * 70)
print("📊 СВОДНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ")
print("=" * 70)
print(f"{'Модель':<25} {'ROC-AUC':>10} {'Accuracy':>10} {'F1 (macro)':>12}")
print("-" * 59)

summary_data = []
for name, preds in predictions.items():
    # Для LAMA используем y_test_lama, для традиционных — y_test
    # (одни и те же строки, одни и те же значения таргета)
    y_true = y_test_lama if 'LightAutoML' in name else y_test

    auc_val = roc_auc_score(y_true, preds)
    pred_classes = (preds >= 0.5).astype(int)
    acc = accuracy_score(y_true, pred_classes)
    f1 = f1_score(y_true, pred_classes, average='macro')
    print(f"   {name:<23} {auc_val:>10.4f} {acc:>10.4f} {f1:>12.4f}")
    summary_data.append({'Model': name, 'ROC-AUC': auc_val, 'Accuracy': acc, 'F1-macro': f1})

summary_df = pd.DataFrame(summary_data).sort_values('ROC-AUC', ascending=False)
print(f"\n🏆 Лучшая: {summary_df.iloc[0]['Model']} (AUC={summary_df.iloc[0]['ROC-AUC']:.4f})")

# 5B. ROC-кривые с CI (bootstrap=500)
def plot_roc_with_ci(y_true, y_pred_proba, model_name, plot_ax, n_bootstraps=500):
    tprs, aucs = [], []
    mean_fpr = np.linspace(0, 1, 100)
    rng = np.random.RandomState(42)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true.iloc[indices])) < 2:
            continue
        y_true_boot = y_true.iloc[indices]
        y_scores_boot = y_pred_proba[indices]
        fpr_boot, tpr_boot, _ = roc_curve(y_true_boot, y_scores_boot)
        aucs.append(auc(fpr_boot, tpr_boot))
        interp_tpr = np.interp(mean_fpr, fpr_boot, tpr_boot)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    auc_lower = np.percentile(aucs, 2.5)
    auc_upper = np.percentile(aucs, 97.5)
    tpr_lower = np.percentile(tprs, 2.5, axis=0)
    tpr_upper = np.percentile(tprs, 97.5, axis=0)

    line = plot_ax.plot(mean_fpr, mean_tpr, lw=2, alpha=0.8,
                        label=f'{model_name} (AUC={mean_auc:.3f} [{auc_lower:.3f}-{auc_upper:.3f}])')
    plot_ax.fill_between(mean_fpr, tpr_lower, tpr_upper, alpha=0.1, color=line[0].get_color())
    print(f"   -> {model_name}: AUC={mean_auc:.4f} [95% CI: {auc_lower:.4f}-{auc_upper:.4f}]")

fig, ax = plt.subplots(figsize=(10, 7))
print("\n📈 ROC-кривые (500 bootstrap итераций):")

for name, preds in predictions.items():
    y_true = y_test_lama if 'LightAutoML' in name else y_test
    plot_roc_with_ci(y_true, preds, name, ax)

ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', label='Random (AUC=0.500)')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves: Traditional Models vs LightAutoML (95% CI)', fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

fig.savefig('roc_curve_final_submission.png', dpi=300, bbox_inches='tight')
print(f"\n✅ График сохранен: 'roc_curve_final_submission.png'")
plt.show()


📈 [Шаг 5] Сводная таблица + ROC-кривые...

📊 СВОДНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ
Модель                       ROC-AUC   Accuracy   F1 (macro)
-----------------------------------------------------------
   Logistic Regression         0.8673     0.8446       0.7464
   Random Forest               0.8744     0.8514       0.7446
   XGBoost                     0.8979     0.8625       0.7813
   LightAutoML (best)          0.9055     0.8674       0.7894

🏆 Лучшая: LightAutoML (best) (AUC=0.9055)

📈 ROC-кривые (500 bootstrap итераций):
   -> Logistic Regression: AUC=0.8672 [95% CI: 0.8625-0.8718]
   -> Random Forest: AUC=0.8744 [95% CI: 0.8696-0.8790]
   -> XGBoost: AUC=0.8979 [95% CI: 0.8942-0.9017]
   -> LightAutoML (best): AUC=0.9054 [95% CI: 0.9015-0.9092]

✅ График сохранен: 'roc_curve_final_submission.png'


In [8]:
# ==========================================
# ШАГ 6: Автовыбор лучшей модели + Инференс
# ==========================================

import pickle

print("\n💾 [Шаг 6] Сохранение и инференс...")

# Автоматический выбор лучшей традиционной модели
all_trad_auc = {}
for name in trained_models:
    all_trad_auc[name] = roc_auc_score(y_test, predictions[name])

best_trad_name = max(all_trad_auc, key=all_trad_auc.get)
best_trad_auc = all_trad_auc[best_trad_name]
print(f"\n🏆 Лучшая традиционная: {best_trad_name} (AUC={best_trad_auc:.4f})")

best_model = trained_models[best_trad_name]['model']
best_X_test = trained_models[best_trad_name]['X_test']

# Сохранение
with open("best_model.pkl", 'wb') as f:
    pickle.dump(best_model, f)
print("   -> ✅ Модель сохранена: best_model.pkl")

# Загрузка
with open("best_model.pkl", 'rb') as f:
    loaded_model = pickle.load(f)
print("   -> ✅ Модель загружена.")

# Инференс 1 записи
if isinstance(best_X_test, np.ndarray):
    sample_record = best_X_test[0:1]
else:
    sample_record = best_X_test.iloc[0:1]
real_value = y_test.iloc[0]

pred_class = loaded_model.predict(sample_record)[0]
pred_proba = loaded_model.predict_proba(sample_record)[:, 1][0]

print(f"\n🔍 Инференс:")
print(f"   -> Предсказанный класс: {pred_class}")
print(f"   -> Вероятность дождя: {pred_proba:.4f}")
print(f"   -> Реальное значение: {real_value}")
print(f"\n{'🎉 Верно!' if pred_class == real_value else '⚠️ Ошибка.'}")


💾 [Шаг 6] Сохранение и инференс...

🏆 Лучшая традиционная: XGBoost (AUC=0.8979)
   -> ✅ Модель сохранена: best_model.pkl
   -> ✅ Модель загружена.

🔍 Инференс:
   -> Предсказанный класс: 0
   -> Вероятность дождя: 0.1153
   -> Реальное значение: 0

🎉 Верно!


In [9]:
# ==========================================
# ШАГ 7: РЕФЛЕКСИЯ
# ==========================================

print("\n")
print("=" * 70)
print("📝 ШАГ 7: РЕФЛЕКСИЯ (Традиционный подход vs AutoML)")
print("=" * 70)

lama_best_auc = best_lama['auc']

print(f"""
1. КАЧЕСТВО МОДЕЛЕЙ:
   • Лучшая традиционная: {best_trad_name} — AUC = {best_trad_auc:.4f}
   • Лучшая LightAutoML:  {best_lama_name} — AUC = {lama_best_auc:.4f}
   • Разница: {abs(lama_best_auc - best_trad_auc):.4f}
   {'→ AutoML лучше.' if lama_best_auc > best_trad_auc else '→ Традиционный подход лучше.'}

2. ПОДГОТОВКА ДАННЫХ:
   • Традиционный: ручная обработка пропусков (медиана/мода),
     ручное кодирование (WoE, TargetEncoder), стандартизация,
     ручной разбор дат (Year, Month, Day)
   • LightAutoML: автоматическая обработка ВСЕГО —
     пропуски, категории, даты, подбор гиперпараметров

3. ТРУДОЗАТРАТЫ:
   • Традиционный: ~3-5 часов (EDA + очистка + кодирование + тюнинг)
   • AutoML: ~15-20 минут (загрузка данных + запуск)

4. ПРЕИМУЩЕСТВА:
   Традиционный:
     ✓ Полный контроль над пайплайном
     ✓ Интерпретируемость (особенно LogReg + WoE)
     ✓ Возможность доменной настройки
   AutoML:
     ✓ Быстрый сильный baseline
     ✓ Автоматический Feature Engineering
     ✓ Автоматический подбор моделей и ансамблирование

5. ВЫВОД:
   Оптимальная стратегия — использовать оба подхода:
   AutoML для быстрого baseline и оценки потенциала данных,
   традиционный для тонкой настройки и интерпретации.
""")
print("=" * 70)
print("✅ ВСЕ ШАГИ ВЫПОЛНЕНЫ! Проект готов к сдаче.")
print("=" * 70)




📝 ШАГ 7: РЕФЛЕКСИЯ (Традиционный подход vs AutoML)

1. КАЧЕСТВО МОДЕЛЕЙ:
   • Лучшая традиционная: XGBoost — AUC = 0.8979
   • Лучшая LightAutoML:  LAMA_Exp3_Utilized_600s — AUC = 0.9055
   • Разница: 0.0076
   → AutoML лучше.

2. ПОДГОТОВКА ДАННЫХ:
   • Традиционный: ручная обработка пропусков (медиана/мода),
     ручное кодирование (WoE, TargetEncoder), стандартизация,
     ручной разбор дат (Year, Month, Day)
   • LightAutoML: автоматическая обработка ВСЕГО —
     пропуски, категории, даты, подбор гиперпараметров

3. ТРУДОЗАТРАТЫ:
   • Традиционный: ~3-5 часов (EDA + очистка + кодирование + тюнинг)
   • AutoML: ~15-20 минут (загрузка данных + запуск)

4. ПРЕИМУЩЕСТВА:
   Традиционный:
     ✓ Полный контроль над пайплайном
     ✓ Интерпретируемость (особенно LogReg + WoE)
     ✓ Возможность доменной настройки
   AutoML:
     ✓ Быстрый сильный baseline
     ✓ Автоматический Feature Engineering
     ✓ Автоматический подбор моделей и ансамблирование

5. ВЫВОД:
   Оптимальная стратегия